# Querying the dynamical.org STAC Catalog with rustac

This notebook shows how to query the dynamical.org STAC catalog stored as a
[stac-geoparquet](https://github.com/stac-utils/stac-geoparquet) file on R2,
using [rustac](https://github.com/stac-utils/rustac) — Python bindings to the
high-performance [stac-rs](https://github.com/stac-utils/stac-rs) Rust library.

Compared to the static-JSON approach (parsing JSON files with `pystac`),
this approach:
- Uses **DuckDB** under the hood for fast column-store queries
- Supports **spatial queries** via bounding box
- Supports **CQL2 filter expressions** for complex queries
- Scales to catalogs with thousands of items without downloading all JSON

## Install dependencies
```
pip install rustac xpystac
```

In [ ]:
import rustac

PARQUET_URL = "https://r2-pub.openscicomp.io/stac/dynamical/catalog.parquet"

client = rustac.DuckdbClient()
print(f"rustac ready — querying {PARQUET_URL}")

## 1. List all items

In [ ]:
items = list(client.search(PARQUET_URL, limit=100))
print(f"{len(items)} items in catalog:\n")
for item in items:
    props = item.get("properties", {})
    print(f"{item['id']}")
    print(f"  bbox:  {item.get('bbox')}")
    print(f"  start: {props.get('start_datetime')}")
    print(f"  end:   {props.get('end_datetime')}")
    print()

## 2. Spatial query — datasets that cover a region

Pass a `bbox` as `[lon_min, lat_min, lon_max, lat_max]`. DuckDB runs the
spatial intersection server-side.

In [ ]:
italy_bbox = [6.6, 36.6, 18.5, 47.1]  # lon_min, lat_min, lon_max, lat_max

matches = list(client.search(PARQUET_URL, bbox=italy_bbox, limit=100))

print(f"Datasets covering Italy ({len(matches)} of {len(items)}):\n")
for item in matches:
    print(f"  {item['id']}")

## 3. Temporal query — datasets covering a date range

For items with `start_datetime` / `end_datetime` (and `datetime: null`),
use a CQL2 `filter` expression to check interval overlap.

The overlap condition is:  `start_datetime <= query_end AND end_datetime >= query_start`

In [ ]:
query_start = "2024-01-01T00:00:00Z"
query_end   = "2025-01-01T00:00:00Z"

time_filter = {
    "op": "and",
    "args": [
        {"op": "<=", "args": [{"property": "start_datetime"}, query_end]},
        {"op": ">=", "args": [{"property": "end_datetime"}, query_start]},
    ],
}

matches_time = list(client.search(PARQUET_URL, filter=time_filter, limit=100))

print(f"Datasets covering {query_start[:10]} – {query_end[:10]} ({len(matches_time)}):\n")
for item in matches_time:
    props = item.get("properties", {})
    print(f"  {item['id']}")
    print(f"    {props.get('start_datetime')} → {props.get('end_datetime')}")

## 4. Combined spatial + temporal query

Combine `bbox` with a CQL2 `filter` to find datasets that cover
both Italy **and** a given date range.

In [ ]:
combined = list(client.search(
    PARQUET_URL,
    bbox=italy_bbox,
    filter=time_filter,
    limit=100,
))

print(f"Datasets covering Italy and {query_start[:10]} – {query_end[:10]} ({len(combined)}):\n")
for item in combined:
    print(f"  {item['id']}")

## 5. Open a matching dataset with xarray

rustac drops non-spec top-level attributes (like `storage:schemes`) when writing
GeoParquet, so we look the item up by ID in the original JSON catalog, which has
the full metadata needed by `xpystac` to open the icechunk store.

In [ ]:
import pystac
import xarray as xr
import xpystac  # registers xarray backend for icechunk assets

CATALOG_URL = "https://r2-pub.openscicomp.io/stac/dynamical/catalog.json"

# Load the JSON catalog once (small — 7 items, ~1 KB)
catalog = pystac.Catalog.from_file(CATALOG_URL)
item_index = {item.id: item for item in catalog.get_items()}

# Pick the first match from the rustac query and fetch the full JSON item
match_id = combined[0]["id"]
item = item_index[match_id]
print(f"Opening: {item.id}")

asset_key = next(k for k in item.assets if "@" in k)
asset = item.assets[asset_key]

ds = xr.open_dataset(asset)
ds